In [1]:
## init mongo db and fiftyone connection
import os

# Define the URI to point to your manual process
os.environ["FIFTYONE_DATABASE_URI"] = "mongodb://localhost:44123"

import fiftyone as fo

# Verify connection
print(fo.core.odm.database.get_db_conn()) 


You are running the oldest supported major version of MongoDB. Please refer to https://deprecation.voxel51.com for deprecation notices. You can suppress this exception by setting your `database_validation` config parameter to `False`. See https://docs.voxel51.com/user_guide/config.html#configuring-a-mongodb-connection for more information
Database(MongoClient(host=['localhost:44123'], document_class=dict, tz_aware=False, connect=True, appname='fiftyone'), 'fiftyone')


In [2]:
import fiftyone.brain as fob
from sklearn.preprocessing import normalize
import plotly.express as px
import skdim
import random
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import ot
from sklearn.manifold import TSNE
import cv2
from fiftyone import ViewField as F
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
import random
import glob 
import math
# ## renders plotly properly in a html instance. 
# import plotly.io as pio
# pio.renderers.default = "notebook"


In [3]:
## Load dataset and views from mongodb 
dataset = fo.load_dataset("dugong")

## load the views
nc_view = dataset.load_saved_view("New_Caledonia")
wp_view = dataset.load_saved_view("West_Papua")

### Repeated Random Sub-sampling Validation

In [ ]:
from sklearn.model_selection import train_test_split
import pandas as pd


def tag_train_test_split_seeded(train_size:float,test_size:float,val_size:float,
                                runs:int,
                                dataset):
    """
    Creates the split for train, test and validation using a stratified pick given by the complexity. 
    Args:
        runs: Number of loop to pass and create the tag inside the dataset 
    
        Returns:
        Return the seed_number and the dataset get tagged.
    """
    # target islands (skipping MANTASANDY)
    islands_to_split = ['UM', 'GAM', 'FRIWEN']
    seeds_list = []

    ## tags adding train_seed_number for west papua 
    for run in range(0,runs+1):
        seed_number  = random.randint(1,50)
        print(f"Seed number selected:{seed_number}")
        seeds_list.append(seed_number)
        for island in islands_to_split:
            # Get a view of just this island
            island_view = dataset.match(F("subregion") == island)
            
            ids = island_view.values("id")

            # Our strata is the complexity (high/medium/low)
            strata = island_view.values("background_complexity")
            
            # Split off the TEST set (20%) ---
            # Stratify ensures the 'high complexity' ratio stays the same
            train_val_ids, test_ids = train_test_split(
                ids, 
                test_size= test_size, 
                stratify=strata,
                shuffle=True, 
                random_state=seed_number
            )
            
            # Get strata for the remaining 80% to split again
            train_val_strata = [s for i, s in zip(ids, strata) if i in train_val_ids]
            
            # Split remaining 80% into Train (70% total) and Val (10% total) ---
            # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
            train_ids, val_ids = train_test_split(
                train_val_ids, 
                test_size= (val_size/(1-test_size)), 
                stratify=train_val_strata, 
                random_state=seed_number
            )
            
            # 3. Apply the tags in FiftyOne
            dataset.select(train_ids).tag_samples(f"train_{str(seed_number)}")
            dataset.select(val_ids).tag_samples(f"val_{str(seed_number)}")
            dataset.select(test_ids).tag_samples(f"test_{str(seed_number)}")
            
            print(f"Island {island}: Train={len(train_ids)}, Test={len(test_ids)}, Val={len(val_ids)}")

    return seeds_list








In [9]:
dataset.distinct("subregion")

['FRIWEN', 'GAM', 'MANTASANDY', 'NC', 'UM']

In [ ]:
## use the function to run 
folder =  '/share/home/e2406743/dataset/exported_img/seed_42'
## percentage of each train, val, test
train_size = 0.7
test_size = 0.2 
val_size = 0.1
runs = 1

seed_number_list = tag_train_test_split_seeded(train_size, test_size, val_size,
                                          runs=runs,
                                          dataset= dataset
                                          )

## tag TRAIN for all new caledonia samples.
nc_view = dataset.match(F("region") == "NC")
nc_view.tag_samples("train")

### delete old tags

In [7]:
dataset.distinct("tags")

[]

# split - train, test, val for full images paths 
## create csv filepaths:

In [9]:
def return_list_filepath_train_test_val(seed_number, dataset, nc_view):
    train_seed_filepath = dataset.match_tags(f"train_{seed_number}").values("filepath")
    test_seed_filepath = dataset.match_tags(f"test_{seed_number}").values("filepath")
    val_seed_filepath = dataset.match_tags(f"val_{seed_number}").values("filepath")
    train_nc_filepath = nc_view.match_tags("train").values("filepath")
    return train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath


def build_filepath_df(train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath):

    df = pd.DataFrame({
        "train_seed": pd.Series(train_seed_filepath),
        "test_seed": pd.Series(test_seed_filepath),
        "val_seed": pd.Series(val_seed_filepath),
        "train_nc": pd.Series(train_nc_filepath),
    })

    return df



## IMPLEMENT LOOP HERE
## RUN ALL GIVEN SEEDS AND CREATES A CSV WITH THE PATHS REGARDING THE FULL IMAGE
for ss in seed_number_list:
    print(f"Running seed:{ss}")
    train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath = return_list_filepath_train_test_val(
        ss, dataset=dataset, nc_view=nc_view
    )

    df_seed = build_filepath_df(
        train_seed_filepath,
        test_seed_filepath,
        val_seed_filepath,
        train_nc_filepath
    )

    ## save it keeping 
    output_filename = f"df_train_test_split_filepath_{str(ss)}.csv"
    print(f"saving file:{output_filename}")
    output_folder = "/share/home/e2406743/dataset/df_filepaths"
    os.makedirs(output_folder, exist_ok=True)
    print(f"saving at:{os.path.join(output_folder,output_filename)}")
    df_seed.to_csv(os.path.join(output_folder, output_filename))
    print('done!')


Running seed:38
saving file:df_train_test_split_filepath_38.csv
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_38.csv
done!
Running seed:40
saving file:df_train_test_split_filepath_40.csv
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_40.csv
done!


## load back

In [5]:
def get_seed_from_filepath(csv_file):
    path = Path(csv_file).stem
    return path.split('_')[-1]

def return_list_from_csv(csv_file):
    dff = pd.read_csv(csv_file)
    wp_train_list = dff['train_seed'].dropna().values
    test_list = dff['test_seed'].dropna().values
    val_list = dff['val_seed'].dropna().values
    nc_train_list = dff['train_nc'].dropna().values
    return wp_train_list, nc_train_list, test_list, val_list

In [14]:
get_seed_from_filepath(csv_file)

'23'

In [11]:
from sklearn.model_selection import train_test_split

In [16]:
csv_file='/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_38.csv'
wp_train_list, nc_train_list, test_list, val_list = return_list_from_csv(csv_file)

ids = np.array(ids)
train_size = 0.7
test_size = 0.2
val_size = 0.1
seed_number = int(get_seed_from_filepath(csv_file))
assert type(seed_number)==int

# Split off the TEST set (20%) ---
# Stratify ensures the 'high complexity' ratio stays the same
train_val_ids, test_ids = train_test_split(
    ids, 
    test_size= test_size, 
    shuffle=True, 
    random_state=seed_number
)


# Split remaining 80% into Train (70% total) and Val (10% total) ---
# 0.125 * 0.8 = 0.1 (which is 10% of the original total)
train_ids, val_ids = train_test_split(
    train_val_ids, 
    test_size= (val_size/(1-test_size)),
    random_state=seed_number
)

In [18]:
def create_train_test_split_for_nc(nc_train_list):
    from sklearn.model_selection import train_test_split
    train_size = 0.7
    test_size = 0.2
    val_size = 0.1
    seed_number = int(get_seed_from_filepath(csv_file))
    assert type(seed_number)==int

    # Split off the TEST set (20%) ---
    # Stratify ensures the 'high complexity' ratio stays the same
    train_val_ids, test_ids = train_test_split(
        ids, 
        test_size= test_size, 
        shuffle=True, 
        random_state=seed_number
    )


    # Split remaining 80% into Train (70% total) and Val (10% total) ---
    # 0.125 * 0.8 = 0.1 (which is 10% of the original total)
    train_ids, val_ids = train_test_split(
        train_val_ids, 
        test_size= (val_size/(1-test_size)),
        random_state=seed_number
    )
    return train_ids, test_ids, val_ids
    

In [19]:
train_list_ncnc, test_list_ncnc, val_list_ncnc = create_train_test_split_for_nc(nc_train_list)

In [20]:
print(len(train_list_ncnc), len(test_list_ncnc), len(val_list_ncnc))

500 144 72


In [ ]:
from sklearn.model_selection import train_test_split

strata = nc_view.values("background_complexity")
            
# Split off the TEST set (20%) ---
# Stratify ensures the 'high complexity' ratio stays the same
train_val_ids, test_ids = train_test_split(
    ids, 
    test_size= test_size, 
    stratify=strata,
    shuffle=True, 
    random_state=seed_number
)

'/share/home/e2406743/dataset/dataset/NC/Flight_226/images/GH034226-619fa2d56d4d3_92.jpeg'

here we have a list containing the filepath of the full images. From these lists, I can randomly select different percentage regarding the whole list size and also using fiftyone with uniqueness. 
After this, it is necessary to set up the mapping dict to map the full images with the associated tiles.

In [13]:
len(wp_train_list)

1423

In [35]:

def random_choice_train_list(train_list,seed,partitions:list=[0.1,0.25,0.5,0.75,1.0]):
    length = len(train_list)

    ## seed 
    random.seed(seed)

    dict_out ={}
    for p in partitions:
        num_images = int(math.floor(length*p))
        dict_out[f"partition_{str(int(p*100))}"] = random.choices(train_list, k=num_images)
    
    return dict_out

In [34]:
## seed with the same seed used to create the partition.
my_seed = random.seed(int(get_seed_from_filepath(csv_file)))

dictt = random_choice_train_list(train_list = wp_train_list,
                                    seed = my_seed,
                                    partitions = [0.05,0.1,0.25,0.5,0.75,1.0]
                                )

NameError: name 'random_choice_train_list' is not defined

In [16]:
len(dictt['partition_25']), len(dictt['partition_75'])

(355, 1067)

In [ ]:
def get_files_by_stem(filepath_stem, patch_folder):
    dict_out = {}
    foolder_meta = os.path.join(patch_folder, 'metadata')
    list_meta = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.json')))
    foolder_meta = os.path.join(patch_folder, 'images')
    list_images = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.jpg')))
    foolder_meta = os.path.join(patch_folder, 'labels')
    list_labels = list(glob.glob(os.path.join(foolder_meta, f'{filepath_stem}__*.txt')))
    dict_out['metadata'] = list_meta
    dict_out['label'] = list_labels
    dict_out['images'] = list_images
    return dict_out

## use the map dict to create the final list of filepaths regarding the patches 
def mapdict_patches_filepath(list_paths, patch_folder):
    dict_map_filepath = {}
    for path in list_paths:
        stem = Path(path).stem
        dict_map_filepath[stem] = get_files_by_stem(stem, patch_folder)

    ## flat dict
    filepath_all_images   = [f for d in dict_map_filepath.values() for f in d.get('images', [])]
    filepath_all_labels   = [f for d in dict_map_filepath.values() for f in d.get('label', [])]
    filepath_all_metadata = [f for d in dict_map_filepath.values() for f in d.get('metadata', [])]

    ## -------------
    print(f'images:{len(filepath_all_images)}')
    print(f"labels:{len(filepath_all_labels)}")
    print(f"metadata:{len(filepath_all_metadata)}")

    return filepath_all_images, filepath_all_labels, filepath_all_metadata

In [18]:
list_images, list_labels, list_metadata = mapdict_patches_filepath(dictt['partition_25'])

images:1089
labels:1089
metadata:1089


# full pipeline

In [18]:
import re 
[match for x in dataset.distinct("tags") for match in re.findall(r"\d+",x)]

['38', '40', '38', '40', '38', '40']

In [14]:
dataset.distinct("tags")


['test_38', 'test_40', 'train', 'train_38', 'train_40', 'val_38', 'val_40']

In [37]:
def get_seed_number_from_tags(tags_list):
    import re 
    return pd.Series([int(match) for x in tags_list for match in re.findall(r"\d+",x)]).unique()


In [30]:
get_seed_number_from_tags(tags_list = dataset.distinct("tags"))

array([38, 40])

In [ ]:
## use the function to run 
csv_file='/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_38.csv'
output_folder = "/share/home/e2406743/dataset/df_filepaths"
patch_folder = "/share/home/e2406743/dataset/exported_img/seed_42"

## this control if it will split and tag the dataset. However, there is no necessity to do it again once its already done.
## so turning false skips this section and look the folder with the already existent csv_filepaths of the FULL IMAGE
split_and_tag  = False
## percentage of each train, val, test
train_size = 0.7
test_size = 0.2 
val_size = 0.1
runs = 1 # number of seeds to create, if 1 is equals to:2. The loop always sums 1 +=

if split_and_tag:
    ## create TAGS in the dataset. train_seednumber, test, val
    seed_number_list = tag_train_test_split_seeded(train_size, test_size, val_size,
                                            runs=runs,
                                            dataset= dataset
                                            )

    ## tag TRAIN for all new caledonia samples.
    nc_view = dataset.match(F("region") == "NC")
    nc_view.tag_samples("train")

    ## get all the unique seeds within the TAGS
    seed_number_list = get_seed_number_from_tags(tags_list = dataset.distinct("tags"))

    ## filter the tag
    ## RUN ALL GIVEN SEEDS AND CREATES A CSV WITH THE PATHS REGARDING THE FULL IMAGE
    for ss in seed_number_list:
        print(f"Running seed:{ss}")
        train_seed_filepath, test_seed_filepath, val_seed_filepath, train_nc_filepath = return_list_filepath_train_test_val(
            ss, dataset=dataset, nc_view=nc_view
        )

        df_seed = build_filepath_df(
            train_seed_filepath,
            test_seed_filepath,
            val_seed_filepath,
            train_nc_filepath
        )

        ## save it keeping 
        output_filename = f"df_train_test_split_filepath_{str(ss)}.csv"
        print(f"saving file:{output_filename}")
        os.makedirs(output_folder, exist_ok=True)
        print(f"saving at:{os.path.join(output_folder,output_filename)}")
        df_seed.to_csv(os.path.join(output_folder, output_filename))
        print('done!')

## --------------------------------------------------
## HERE IT IS BEING DOING FOR ONE SEED THAT WAS TAGGED IN TAGS
#### LOAD CSV FILEPATH BACK 

wp_train_list, nc_train_list, test_list, val_list = return_list_from_csv(csv_file)

## seed with the same seed used to create the partition.
my_seed = int(get_seed_from_filepath(csv_file))
print(f"seeding :{my_seed}")
random.seed(my_seed)

## TRAIN WP--------------------
## create a dict containing the filepath_stem with dict keys containig the filepath for images, labels and metadata
dictt = random_choice_train_list(train_list = wp_train_list,
                                    seed = my_seed,
                                    partitions = [0.05,0.1,0.25,0.5,0.75,1.0]
                                )


## BEST SET 
## make a functoin to choose the best set of images 
## TODO

## LOOP into each dictt key and return a full dataset
## return for each partition the paths associated for images, labels and metadata.

new_dict = dictt.copy()
output_dict_partitions = dict()

for key in new_dict.keys():
    print(f"Running key:{key}")

    ## retriveves for each partition the associated patches, labels, metadata  - filepath 
    list_images, list_labels, list_metadata = mapdict_patches_filepath(dictt[key], patch_folder)
    output_dict_partitions[key] = {'images':list_images , 'labels':list_labels, 'metadata': list_metadata}

## TRAIN WP- SAVE DF 
print("\n")
print("saving patches filepath with the partition and selected by the given seed")
df_patches_filepath = pd.DataFrame().from_dict(output_dict_partitions)
output_filename = f"df_train_test_split_filepath_PATCHES_wpartitions_seed_{str(my_seed)}.parquet"
print(f"saving file:{output_filename}")
os.makedirs(output_folder, exist_ok=True)
print(f"saving at:{os.path.join(output_folder,output_filename)}")
df_patches_filepath.to_parquet((os.path.join(output_folder, output_filename)))



Running key:partition_5
images:261
labels:261
metadata:261
Running key:partition_10
images:464
labels:464
metadata:464
Running key:partition_25
images:1085
labels:1085
metadata:1085
Running key:partition_50
images:1973
labels:1973
metadata:1973
Running key:partition_75
images:2602
labels:2602
metadata:2602
Running key:partition_100
images:3087
labels:3087
metadata:3087


saving patches filepath with the partition and selected by the given seed
saving file:df_train_test_split_filepath_PATCHES_wpartitions_seed_38.parquet
saving at:/share/home/e2406743/dataset/df_filepaths/df_train_test_split_filepath_PATCHES_wpartitions_seed_38.parquet


In [ ]:
## TEST VAL AND NCTRAIN
## map into the fiepath
print(f"mapping train:")
nc_train_list_images, nc_train_list_labels, nc_train_list_metadata = mapdict_patches_filepath(nc_train_list,
                                                                                               patch_folder
                                                                                               )

print(f"mapping test")
test_list_images, test_list_labels, test_list_metadata = mapdict_patches_filepath(test_list,
                                                                                               patch_folder
                                                                                               )

print(f"mapping val")
val_list_images, val_list_labels, val_list_metadata = mapdict_patches_filepath(val_list,
                                                                                               patch_folder
                                                                                               )

images:3027
labels:3027
metadata:3027
images:1382
labels:1382
metadata:1382
images:723
labels:723
metadata:723


In [ ]:
def produce_dict_for_dataset_pytorch(nc_train_list,
                                     )

In [52]:
df_patches_filepath = pd.DataFrame().from_dict(output_dict_partitions)

In [63]:
print(df_patches_filepath.columns)

Index(['partition_5', 'partition_10', 'partition_25', 'partition_50',
       'partition_75', 'partition_100'],
      dtype='str')


In [60]:
df_patches_filepath.loc['images']['partition_5']

['/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M11_F3_GSP_DJI_0009-66acd97abe6b7_228__tile_540_2700_n.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M11_F3_GSP_DJI_0009-66acd97abe6b7_228__tile_0_2160_p.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M11_F3_GSP_DJI_0009-66acd97abe6b7_228__tile_540_2160_p.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M8_F3_GSP_DJI_0007-66c891c6e484f_12__tile_1080_540_p.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M8_F3_GSP_DJI_0007-66c891c6e484f_12__tile_540_540_p.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M8_F3_GSP_DJI_0007-66c891c6e484f_12__tile_1080_1080_p.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M8_F3_GSP_DJI_0007-66c891c6e484f_12__tile_0_1620_n.jpg',
 '/share/home/e2406743/dataset/exported_img/seed_42/images/MAN_P4_FRIWEN_M

In [61]:
df_patches_filepath.index

Index(['images', 'labels', 'metadata'], dtype='str')